# Machine Learning
In this notebook, we will be training Machine Learning models to predict the default flag of a given loan. We will be creating a pseudo-label using the loan_status column by classifying extremely late, charged off, and default loans as having a True default flag. We will be using Spark ML to train our models to take advantage of distributed computation and make our approach scaleable for big data.

We will be training two separate models in this phase of our project, known as the dev phase. Based on their performance on an unseen dataset in the next phase, one of these two models will be selected for inference on entirely new data.

#### Import and Read the data

In [0]:
spark.conf.set(
    "fs.azure.account.key.lcdevadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcstgadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcprodadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lendingclubstorage.blob.core.windows.net",
    "REDACTED"
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, lit, sum
import pyspark.sql.functions as F

# INPUT_PATH = "../dataset/processed"
INPUT_PATH = "wasbs://lcloansdev@lcdevadls.blob.core.windows.net/"

df = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

#### Create the default_flag Column

In [0]:
# For default flag
df = df.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

#### Observe any Class Imabalances

In [0]:
# Let's also check the overall default rate
df.agg(
    count("*").alias("total_loans"),
    sum("default_flag").alias("total_defaults"),
).withColumn(
    "default_rate",
    when(col("total_loans") != 0, col("total_defaults") / col("total_loans")).otherwise(None)
).show()

We have a significant class imbalance in our dataset as we have many more non-default loans as we do defaults. To mitigate this, we will be taking two steps:
- We will be using the professional technique of using Class Weights. The minority class will be assigned a higher weight during training, allowing us to train the model properly. This technique is superior to more naive techniques like oversampling the minority task as it retains the original distribution.
- We won't rely on accuracy as a tool to evaluate our models. Due to this class imabalance (which plausibly also exists in the testing dataset), accuracy will be misleading as the model could always predict the majority class and still score a >95% accuracy. 
- Instead of accuracy, we will be looking at the AUC-ROC value which measures how well the model ranks positives vs negatives, and is known to be great for binary classification. 
- While AUC-ROC will be our main focus, we will also report F1 score (another good metric in this case) and accuracy.

#### Compute Class Weights

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import col, when

# Count of default and non-default
counts = df.groupBy("default_flag").count().collect()
count_dict = {row["default_flag"]: row["count"] for row in counts}
total = __builtins__.sum(count_dict.values())
weight_default = total / (2 * count_dict.get(1, 1))
weight_non_default = total / (2 * count_dict.get(0, 1))

# Create a weight column
df = df.withColumn(
    "class_weight",
    when(col("default_flag") == 1, weight_default).otherwise(weight_non_default)
)

#### Prepare Features
- Select the Numeric and Categorical features we want to use
- Index the categorical features for training
- Caste the numeric columns
- Combine the features using VectorAssembler
- Scale numeric features using StandardScaler

In [0]:
numeric_cols = ["loan_amnt", "funded_amnt", "funded_amnt_inv", "int_rate", "installment",
                "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
                "revol_util", "total_acc", "acc_open_past_24mths", "bc_open_to_buy",
                "bc_util", "chargeoff_within_12_mths", "delinq_amnt"]

categorical_cols = ["term", "grade", "sub_grade", "home_ownership", "verification_status", "purpose", "addr_state"]

from pyspark.sql.functions import isnan

# Drop rows where numeric columns are NaN
for col_name in categorical_cols:
    df = df.filter(~isnan(col(col_name)))

# Index categorical features
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[c+"_idx" for c in categorical_cols],
                        outputCols=[c+"_ohe" for c in categorical_cols],
                        handleInvalid="keep")

# caste all numeric columns to float
for col_name in numeric_cols:
    df = df.withColumn(col_name, col(col_name).cast("float"))


# Assemble all features
assembler = VectorAssembler(
    inputCols=numeric_cols + [c+"_ohe" for c in categorical_cols],
    outputCol="features",
        handleInvalid="keep"
)

# Scale numeric features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

# Create Train-Test Split

# Create Train-Test Split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [0]:
df = df.withColumn("revol_util", F.regexp_replace("revol_util", "%", ""))


In [0]:
df = df.withColumn("bc_util",
                   when((F.col("bc_util") == "") | (F.col("bc_util") == "NA"), None)
                   .otherwise(F.col("bc_util")))
df = df.withColumn("bc_util", df["bc_util"].cast("float"))


In [0]:
for c in numeric_cols:
    df = df.withColumn(c, when(col(c) == "", None).otherwise(col(c)))
    df = df.withColumn(c, col(c).cast("float"))


In [0]:
for c in numeric_cols:
    mean_val = df.select(F.mean(col(c))).first()[0]
    df = df.fillna({c: mean_val})


In [0]:
import pyspark.sql.functions as F
numeric_cols = [col for col, dtype in df.dtypes if dtype in ('int', 'double', 'float', 'long')]
for c in numeric_cols:
    print(c, df.filter(F.isnan(c) | F.col(c).isNull()).count())


#### Model 1: Logistic Regression
##### Training

In [0]:
lr = LogisticRegression(featuresCol="scaled_features",
                        labelCol="default_flag",
                        weightCol="class_weight",
                        maxIter=100,
                        regParam=0.01)

# Create the Pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler, lr])

# Fit the model
model1 = pipeline.fit(train_df)


##### Evaluation

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Make predictions
predictions = model1.transform(test_df)

# BinaryClassificationEvaluator for AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="default_flag", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc_roc = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc_roc:.4f}")

# MulticlassClassificationEvaluator for F1
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="f1"
)
f1_score = evaluator_f1.evaluate(predictions)
print(f"F1-score: {f1_score:.4f}")

# Accuracy (not ideal for class imbalance, but can report)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

#### Model 2: Random Forest Classifier
##### Training

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="default_flag",
    weightCol="class_weight", 
    numTrees=50,
    maxDepth=5,
    seed=42
)

# Create the Pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler, rf])

# Fit the model
model2 = pipeline.fit(train_df)

##### Evaluation

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Make predictions
predictions = model2.transform(test_df)

# BinaryClassificationEvaluator for AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="default_flag", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc_roc = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc_roc:.4f}")

# MulticlassClassificationEvaluator for F1
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="f1"
)
f1_score = evaluator_f1.evaluate(predictions)
print(f"F1-score: {f1_score:.4f}")

# Accuracy (not ideal for class imbalance, but can report)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

#### Save the Models

In [0]:
from pyspark.ml.linalg import DenseVector, SparseVector
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import mlflow
import mlflow.spark
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

# UDF to convert vector to string
def vector_to_string(v):
    if isinstance(v, (DenseVector, SparseVector)):
        return ",".join([str(x) for x in v.toArray()])
    return None

vector_to_string_udf = udf(vector_to_string, StringType())

input_schema = Schema(
    [ColSpec("double", c) for c in numeric_cols] +
    [ColSpec("string", c) for c in categorical_cols]
)

output_schema = Schema([
    ColSpec("double", "prediction"),
    ColSpec("string", "probability"),
    ColSpec("string", "rawPrediction")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="loan_pipeline_model"):
    mlflow.spark.log_model(
        spark_model=model1,
        artifact_path="loan_pipeline_model",
        registered_model_name="LC_Loan_Pipeline",
        signature=signature
    )
